In [ ]:
# import os

# if os.path.exists("NovaS.pdf"):
#     print("blabla")

blabla


In [2]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import OpenAIEmbeddings

# Step -0 : Load PDF into text

In [3]:
text_data = PyPDFLoader("NovaS.pdf").load()
for page in text_data:
    page.metadata["source"] = "NovaS.pdf"

text_data


[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by d

# Step -1 :  Creating Chunks of the text data

In [ ]:
# len(chunks)

101

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.split_documents(text_data)
chunks
# len(chunks)


[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='and technology company that has grown gradually over the years. The organization was'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'moddate': '

# STEP-2 : Creating embeddings of the chunks (optional)

In [4]:
# from langchain_openai import OpenAIEmbeddings

#(optional)

embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

embedded_chunks = embed_model.embed_documents([i.page_content for i in chunks])
embedded_chunks[0]

[-0.0184783935546875,
 -0.0271453857421875,
 -0.0162811279296875,
 0.002834320068359375,
 0.03887939453125,
 -0.029754638671875,
 -0.01235198974609375,
 0.0170135498046875,
 0.000335693359375,
 0.0004940032958984375,
 0.029327392578125,
 0.0240020751953125,
 -0.01898193359375,
 -0.05126953125,
 0.0099029541015625,
 0.041595458984375,
 -0.0174102783203125,
 -0.00591278076171875,
 0.025970458984375,
 -0.0065460205078125,
 -0.033599853515625,
 0.03326416015625,
 -0.0152435302734375,
 0.00560760498046875,
 -0.00379180908203125,
 0.03076171875,
 -0.0152587890625,
 0.037567138671875,
 0.02435302734375,
 -0.039031982421875,
 0.03192138671875,
 -0.02374267578125,
 -0.01068878173828125,
 0.0248870849609375,
 0.0272369384765625,
 0.036895751953125,
 0.01332855224609375,
 -0.006862640380859375,
 0.005908966064453125,
 0.0570068359375,
 0.0026397705078125,
 -0.0023670196533203125,
 0.047821044921875,
 0.06182861328125,
 0.056976318359375,
 0.0213775634765625,
 -0.032806396484375,
 -0.0354614257812

# STEP-3 : Creating embeddings and storing them in Vector DB

In [5]:
from langchain_community.vectorstores import Chroma

embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma.from_documents(chunks, embed_model, persist_directory="./chroma_db")

# Step-4 : Connection & Retrieval

In [6]:
chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)

C:\Users\Hp\AppData\Local\Temp\ipykernel_13116\3816121341.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)


In [7]:
chroma_db_con.similarity_search("When NovaSphere Technologies was found", k=3)

[Document(metadata={'total_pages': 3, 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'author': 'Ansh Lamba', 'page': 0, 'moddate': '2026-03-31T11:24:15-03:00', 'page_label': '1', 'producer': 'Microsoft® Word for Microsoft 365'}, page_content='In 2018, NovaSphere Technologies started getting more attention in the local technology'),
 Document(metadata={'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'page_label': '1', 'creator': 'Microsoft® Word for Microsoft 365', 'page': 0, 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'producer': 'Microsoft® Word for Microsoft 365', 'total_pages': 3}, page_content='In 2018, NovaSphere Technologies started getting more attention in the local technology'),
 Document(metadata={'source': 'NovaS.pdf', 'producer': 'Microsoft® Word for Microsoft 365', 'total_pages': 3, 'creationdate': '2026-03-31T11:24:15-03:00', 'creator': 'Microsoft® Word for Microso

In [8]:
chroma_db_con.similarity_search("By 2019, how many employees were there?", k=2)

[Document(metadata={'page': 0, 'total_pages': 3, 'source': 'NovaS.pdf', 'creator': 'Microsoft® Word for Microsoft 365', 'moddate': '2026-03-31T11:24:15-03:00', 'producer': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'page_label': '1'}, page_content='By the beginning of 2019, the organization had grown to more than fifteen employees. This'),
 Document(metadata={'creator': 'Microsoft® Word for Microsoft 365', 'producer': 'Microsoft® Word for Microsoft 365', 'source': 'NovaS.pdf', 'total_pages': 3, 'author': 'Ansh Lamba', 'page_label': '2', 'moddate': '2026-03-31T11:24:15-03:00', 'page': 1, 'creationdate': '2026-03-31T11:24:15-03:00'}, page_content='During 2023, the organization experienced steady growth. The number of employees')]

# Step -5 : LLM Itegration and answer Generation

In [ ]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
# llm.invoke("What is the name of complany?")

AIMessage(content="I'm sorry, could you please provide more context or details so I can accurately determine the name of the company you are referring to?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 15, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DiC0MLpiPoGE4sVyMgIIQEXvuiWwu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e4dfd-ecdf-7d72-a697-239e0c21de62-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 27, 'total_tokens': 42, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
user_query = input("Enter your Question: ")

rel_chunks = chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)
rel_chunks_content = str(re.l_chunks_content)

llm.invoke(f"{user_query}, Use the following context to answer the question: {rel_chunks_content}")

AIMessage(content='By the beginning of 2019, the organization had grown to more than fifteen employees. This means that there were at least 16 employees by 2019.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 85, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DiCO0cW3rmpEAAmcAKtpGIQq5vd3D', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e4e14-4d91-7030-956e-e69b34b3ef78-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'output_tokens': 33, 'total_tokens': 118, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})